In [1]:
# -*- coding: utf-8 -*-
"""
User Manager (Tkinter) para config.yaml no padrão streamlit-authenticator.

Requisitos:
  pip install pyyaml bcrypt

Uso:
  python user_manager_tk.py
"""

from __future__ import annotations

import os
import re
import time
import secrets
import string
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

import yaml
import bcrypt


# ----------------------------
# Utilitários
# ----------------------------

USERNAME_RE = re.compile(r"^[A-Za-z0-9_.-]{2,32}$")


def now_stamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S")


def strong_password(length: int = 16) -> str:
    """
    Gera senha forte com:
      - letras maiúsculas e minúsculas
      - dígitos
      - símbolos seguros
    Evita espaços e caracteres confusos.
    """
    if length < 12:
        length = 12

    upper = string.ascii_uppercase
    lower = string.ascii_lowercase
    digits = string.digits
    symbols = "!@#$%&*_-+=?"

    pwd = [
        secrets.choice(upper),
        secrets.choice(lower),
        secrets.choice(digits),
        secrets.choice(symbols),
    ]

    pool = upper + lower + digits + symbols
    pwd.extend(secrets.choice(pool) for _ in range(length - len(pwd)))
    secrets.SystemRandom().shuffle(pwd)
    return "".join(pwd)


def bcrypt_hash_password(plain: str) -> str:
    hashed = bcrypt.hashpw(plain.encode("utf-8"), bcrypt.gensalt(rounds=12))
    return hashed.decode("utf-8")


def normalize_yaml_structure(data: dict) -> dict:
    """
    Garante que existam as chaves esperadas.
    Mantém tudo o que já existe no YAML.
    """
    if not isinstance(data, dict):
        data = {}

    data.setdefault("credentials", {})
    data["credentials"].setdefault("usernames", {})

    data.setdefault("preauthorized", {})
    data["preauthorized"].setdefault("emails", [])

    return data


def safe_email(s: str) -> bool:
    s = (s or "").strip()
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", s))


def safe_role(s: str) -> bool:
    return (s or "").strip() in {"admin", "user", "viewer", "gestor", "analista"}


# ----------------------------
# Camada de Dados (YAML)
# ----------------------------

class ConfigStore:
    def __init__(self) -> None:
        self.path: str | None = None
        self.data: dict = normalize_yaml_structure({})

        # Senhas visíveis apenas em memória para exibição na tabela
        # Não são gravadas no YAML
        self.visible_passwords: dict[str, str] = {}

    def load(self, path: str) -> None:
        with open(path, "r", encoding="utf-8") as f:
            loaded = yaml.safe_load(f) or {}

        self.data = normalize_yaml_structure(loaded)
        self.path = path

        # Ao abrir um YAML, não há como recuperar senha em texto puro
        # pois o arquivo só contém hash.
        self.visible_passwords = {}

    def backup(self) -> str:
        if not self.path:
            raise RuntimeError("Nenhum arquivo carregado.")
        bak_path = f"{self.path}.bak_{now_stamp()}"
        with open(self.path, "rb") as src, open(bak_path, "wb") as dst:
            dst.write(src.read())
        return bak_path

    def save(self, path: str | None = None) -> None:
        if path is None:
            if not self.path:
                raise RuntimeError("Nenhum caminho de destino definido.")
            path = self.path

        if self.path and os.path.abspath(path) == os.path.abspath(self.path) and os.path.exists(self.path):
            self.backup()

        with open(path, "w", encoding="utf-8") as f:
            yaml.safe_dump(
                self.data,
                f,
                sort_keys=False,
                allow_unicode=True,
                default_flow_style=False,
                width=120,
            )
        self.path = path

    @property
    def users(self) -> dict:
        return self.data["credentials"]["usernames"]

    def get_user(self, username: str) -> dict | None:
        return self.users.get(username)

    def get_visible_password(self, username: str) -> str:
        return self.visible_passwords.get(username, "")

    def set_visible_password(self, username: str, plain_password: str) -> None:
        self.visible_passwords[username] = plain_password

    def clear_visible_password(self, username: str) -> None:
        if username in self.visible_passwords:
            del self.visible_passwords[username]

    def upsert_user(
        self,
        username: str,
        name: str,
        email: str,
        role: str,
        password_hash: str | None = None,
    ) -> None:
        u = self.users.get(username, {})
        u["name"] = name
        u["email"] = email
        u["role"] = role
        if password_hash:
            u["password"] = password_hash
        self.users[username] = u

    def delete_user(self, username: str) -> None:
        if username in self.users:
            del self.users[username]
        self.clear_visible_password(username)

    def set_password(self, username: str, password_hash: str) -> None:
        if username not in self.users:
            raise KeyError("Usuário não existe.")
        self.users[username]["password"] = password_hash


# ----------------------------
# Interface Tkinter
# ----------------------------

class App(tk.Tk):
    def __init__(self) -> None:
        super().__init__()
        self.title("Gerenciador de Usuários (config.yaml)")
        self.geometry("1180x560")
        self.minsize(1080, 520)

        self.store = ConfigStore()

        self._build_ui()
        self._set_status("Abra um config.yaml para começar.")

    def _build_ui(self) -> None:
        self.columnconfigure(0, weight=1)
        self.rowconfigure(1, weight=1)

        # Top bar
        top = ttk.Frame(self, padding=(10, 10, 10, 6))
        top.grid(row=0, column=0, sticky="ew")
        top.columnconfigure(3, weight=1)

        ttk.Button(top, text="📂 Abrir YAML", command=self.on_open).grid(row=0, column=0, padx=(0, 6))
        ttk.Button(top, text="💾 Salvar", command=self.on_save).grid(row=0, column=1, padx=(0, 6))
        ttk.Button(top, text="💾 Salvar como...", command=self.on_save_as).grid(row=0, column=2, padx=(0, 12))

        ttk.Label(top, text="Filtro:").grid(row=0, column=3, sticky="e")
        self.var_filter = tk.StringVar()
        ent_filter = ttk.Entry(top, textvariable=self.var_filter, width=28)
        ent_filter.grid(row=0, column=4, sticky="e")
        ent_filter.bind("<KeyRelease>", lambda e: self.refresh_table())

        # Main area
        main = ttk.Frame(self, padding=(10, 0, 10, 10))
        main.grid(row=1, column=0, sticky="nsew")
        main.columnconfigure(0, weight=4)
        main.columnconfigure(1, weight=2)
        main.rowconfigure(0, weight=1)

        # Left: table
        left = ttk.Frame(main)
        left.grid(row=0, column=0, sticky="nsew", padx=(0, 10))
        left.rowconfigure(0, weight=1)
        left.columnconfigure(0, weight=1)

        cols = ("username", "name", "email", "role", "generated_password")
        self.tree = ttk.Treeview(left, columns=cols, show="headings", height=18)

        col_specs = {
            "username": (140, "USERNAME"),
            "name": (180, "NOME"),
            "email": (240, "EMAIL"),
            "role": (100, "ROLE"),
            "generated_password": (220, "SENHA GERADA"),
        }

        for c in cols:
            width, title = col_specs[c]
            self.tree.heading(c, text=title)
            self.tree.column(c, width=width, anchor="w")

        self.tree.grid(row=0, column=0, sticky="nsew")

        vsb = ttk.Scrollbar(left, orient="vertical", command=self.tree.yview)
        self.tree.configure(yscrollcommand=vsb.set)
        vsb.grid(row=0, column=1, sticky="ns")

        self.tree.bind("<<TreeviewSelect>>", self.on_select_user)

        # Right: form
        right = ttk.LabelFrame(main, text="Editar / Criar Usuário", padding=10)
        right.grid(row=0, column=1, sticky="nsew")
        right.columnconfigure(1, weight=1)

        self.var_username = tk.StringVar()
        self.var_name = tk.StringVar()
        self.var_email = tk.StringVar()
        self.var_role = tk.StringVar(value="user")
        self.var_password = tk.StringVar()

        r = 0
        ttk.Label(right, text="Username").grid(row=r, column=0, sticky="w", pady=3)
        ttk.Entry(right, textvariable=self.var_username).grid(row=r, column=1, sticky="ew", pady=3)

        r += 1
        ttk.Label(right, text="Name").grid(row=r, column=0, sticky="w", pady=3)
        ttk.Entry(right, textvariable=self.var_name).grid(row=r, column=1, sticky="ew", pady=3)

        r += 1
        ttk.Label(right, text="Email").grid(row=r, column=0, sticky="w", pady=3)
        ttk.Entry(right, textvariable=self.var_email).grid(row=r, column=1, sticky="ew", pady=3)

        r += 1
        ttk.Label(right, text="Role").grid(row=r, column=0, sticky="w", pady=3)
        role_combo = ttk.Combobox(
            right,
            textvariable=self.var_role,
            values=["admin", "user", "viewer", "gestor", "analista"],
            state="readonly",
        )
        role_combo.grid(row=r, column=1, sticky="ew", pady=3)

        r += 1
        ttk.Separator(right).grid(row=r, column=0, columnspan=2, sticky="ew", pady=10)

        r += 1
        ttk.Label(right, text="Nova senha").grid(row=r, column=0, sticky="w", pady=3)
        ttk.Entry(right, textvariable=self.var_password).grid(row=r, column=1, sticky="ew", pady=3)

        r += 1
        btns_pwd = ttk.Frame(right)
        btns_pwd.grid(row=r, column=0, columnspan=2, sticky="ew", pady=(6, 0))
        btns_pwd.columnconfigure(0, weight=1)
        btns_pwd.columnconfigure(1, weight=1)

        ttk.Button(btns_pwd, text="🔑 Gerar senha", command=self.on_generate_password).grid(
            row=0, column=0, sticky="ew", padx=(0, 6)
        )
        ttk.Button(btns_pwd, text="📋 Copiar senha", command=self.on_copy_password).grid(
            row=0, column=1, sticky="ew"
        )

        r += 1
        ttk.Separator(right).grid(row=r, column=0, columnspan=2, sticky="ew", pady=10)

        r += 1
        actions = ttk.Frame(right)
        actions.grid(row=r, column=0, columnspan=2, sticky="ew")
        actions.columnconfigure(0, weight=1)
        actions.columnconfigure(1, weight=1)

        ttk.Button(actions, text="✅ Inserir/Atualizar", command=self.on_upsert).grid(
            row=0, column=0, sticky="ew", padx=(0, 6)
        )
        ttk.Button(actions, text="🗑️ Excluir", command=self.on_delete).grid(
            row=0, column=1, sticky="ew"
        )

        r += 1
        ttk.Button(right, text="🧹 Limpar campos", command=self.clear_form).grid(
            row=r, column=0, columnspan=2, sticky="ew", pady=(10, 0)
        )

        # Status bar
        self.status = tk.StringVar()
        bar = ttk.Frame(self, padding=(10, 6))
        bar.grid(row=2, column=0, sticky="ew")
        ttk.Label(bar, textvariable=self.status).grid(row=0, column=0, sticky="w")

    def _set_status(self, msg: str) -> None:
        self.status.set(msg)

    # ----------------------------
    # Ações de arquivo
    # ----------------------------

    def on_open(self) -> None:
        path = filedialog.askopenfilename(
            title="Abrir config.yaml",
            filetypes=[("YAML", "*.yaml *.yml"), ("Todos", "*.*")],
        )
        if not path:
            return
        try:
            self.store.load(path)
            self.refresh_table()
            self.clear_form()
            self._set_status(
                f"Arquivo carregado: {path} | senhas visíveis antigas não podem ser recuperadas do YAML."
            )
        except Exception as e:
            messagebox.showerror("Erro ao abrir", str(e))

    def on_save(self) -> None:
        if not self.store.path:
            return self.on_save_as()
        try:
            self.store.save()
            self._set_status(f"Salvo com backup automático: {self.store.path}")
        except Exception as e:
            messagebox.showerror("Erro ao salvar", str(e))

    def on_save_as(self) -> None:
        path = filedialog.asksaveasfilename(
            title="Salvar config.yaml como...",
            defaultextension=".yaml",
            filetypes=[("YAML", "*.yaml *.yml")],
        )
        if not path:
            return
        try:
            self.store.save(path)
            self._set_status(f"Salvo em: {path}")
        except Exception as e:
            messagebox.showerror("Erro ao salvar", str(e))

    # ----------------------------
    # Tabela
    # ----------------------------

    def refresh_table(self) -> None:
        for iid in self.tree.get_children():
            self.tree.delete(iid)

        flt = (self.var_filter.get() or "").strip().lower()

        for username, u in sorted(self.store.users.items(), key=lambda kv: kv[0].lower()):
            name = (u.get("name") or "")
            email = (u.get("email") or "")
            role = (u.get("role") or "")
            visible_pwd = self.store.get_visible_password(username)

            row_text = f"{username} {name} {email} {role} {visible_pwd}".lower()

            if flt and flt not in row_text:
                continue

            self.tree.insert(
                "",
                "end",
                values=(username, name, email, role, visible_pwd),
            )

    def on_select_user(self, _evt=None) -> None:
        sel = self.tree.selection()
        if not sel:
            return

        values = self.tree.item(sel[0], "values")
        if not values:
            return

        username = values[0]
        u = self.store.get_user(username)
        if not u:
            return

        self.var_username.set(username)
        self.var_name.set(u.get("name", ""))
        self.var_email.set(u.get("email", ""))
        self.var_role.set(u.get("role", "user"))

        # Agora mostra no campo a senha visível armazenada em memória
        self.var_password.set(self.store.get_visible_password(username))

        self._set_status(f"Selecionado: {username}")


    # ----------------------------
    # Formulário / CRUD
    # ----------------------------

    def clear_form(self) -> None:
        self.var_username.set("")
        self.var_name.set("")
        self.var_email.set("")
        self.var_role.set("user")
        self.var_password.set("")

    def validate_form(self) -> tuple[bool, str]:
        username = (self.var_username.get() or "").strip()
        name = (self.var_name.get() or "").strip()
        email = (self.var_email.get() or "").strip()
        role = (self.var_role.get() or "").strip()

        if not username or not USERNAME_RE.match(username):
            return False, "Username inválido. Use 2–32 chars: letras, números, _ . -"
        if not name:
            return False, "Name é obrigatório."
        if email and not safe_email(email):
            return False, "Email inválido."
        if not safe_role(role):
            return False, "Role inválida (use uma da lista)."

        return True, ""

    def on_upsert(self) -> None:
        ok, msg = self.validate_form()
        if not ok:
            messagebox.showwarning("Validação", msg)
            return

        username = self.var_username.get().strip()
        name = self.var_name.get().strip()
        email = self.var_email.get().strip()
        role = self.var_role.get().strip()
        new_pwd = (self.var_password.get() or "").strip()

        try:
            pwd_hash = None
            if new_pwd:
                pwd_hash = bcrypt_hash_password(new_pwd)

            existed = self.store.get_user(username) is not None
            self.store.upsert_user(
                username,
                name,
                email,
                role,
                password_hash=pwd_hash,
            )

            # Guarda a senha visível em memória para aparecer na tabela
            if new_pwd:
                self.store.set_visible_password(username, new_pwd)

            self.refresh_table()

            self._set_status(
                f"{'Atualizado' if existed else 'Criado'}: {username}"
                + (" (senha atualizada e exibida na tabela)" if new_pwd else "")
            )

        except Exception as e:
            messagebox.showerror("Erro", str(e))

    def on_delete(self) -> None:
        username = (self.var_username.get() or "").strip()
        if not username:
            messagebox.showinfo("Excluir", "Selecione um usuário na lista ou preencha o username.")
            return

        if self.store.get_user(username) is None:
            messagebox.showinfo("Excluir", "Usuário não encontrado.")
            return

        if not messagebox.askyesno("Confirmação", f"Excluir o usuário '{username}'?"):
            return

        try:
            self.store.delete_user(username)
            self.refresh_table()
            self.clear_form()
            self._set_status(f"Excluído: {username}")
        except Exception as e:
            messagebox.showerror("Erro", str(e))

    # ----------------------------
    # Senhas
    # ----------------------------

    def on_generate_password(self) -> None:
        pwd = strong_password(16)
        self.var_password.set(pwd)

        username = (self.var_username.get() or "").strip()
        if username:
            self.store.set_visible_password(username, pwd)

        self.refresh_table()
        self._set_status("Senha gerada. Use 'Inserir/Atualizar' para salvar o hash no YAML.")

    def on_copy_password(self) -> None:
        pwd = (self.var_password.get() or "").strip()
        if not pwd:
            messagebox.showinfo("Copiar", "Não há senha no campo para copiar.")
            return

        self.clipboard_clear()
        self.clipboard_append(pwd)
        self._set_status("Senha copiada para a área de transferência.")


if __name__ == "__main__":
    App().mainloop()
